In [ ]:
import json

# --- Define Available Response Actions ---
# In a real SOAR, these would map to API calls to integrated tools.
RESPONSE_ACTIONS = {
    'block_ip': {'name': 'Block IP Address', 'impact': 'High', 'requires_approval': True},
    'isolate_host': {'name': 'Isolate Host', 'impact': 'High', 'requires_approval': True},
    'disable_user': {'name': 'Disable User Account', 'impact': 'Medium', 'requires_approval': True},
    'reset_password': {'name': 'Reset User Password', 'impact': 'Medium', 'requires_approval': False},
    'scan_host': {'name': 'Scan Host for Malware', 'impact': 'Low', 'requires_approval': False},
    'notify_manager': {'name': 'Notify User Manager', 'impact': 'Low', 'requires_approval': False},
    'create_ticket': {'name': 'Create Investigation Ticket', 'impact': 'Low', 'requires_approval': False},
    'quarantine_file': {'name': 'Quarantine File', 'impact': 'High', 'requires_approval': True}
}

# --- AI-Driven Decision Engine ---
def decide_response_actions(enriched_alert_data):
    print("\n--- AI Decision Engine Activated ---")
    
    # Initialize scores and recommendations
    action_scores = {action: 0 for action in RESPONSE_ACTIONS}
    recommendations = []
    priority = enriched_alert_data.get('priority', 'Low') # Default priority
    action_recommendations = []

    # --- Scoring Logic based on enriched data ---
    # This is where AI models (ML classifiers, expert systems) would provide nuanced scores.
    # For this example, we use heuristic rules based on the data.

    # Base score from alert priority
    if priority == 'Critical':
        action_scores['block_ip'] += 20
        action_scores['isolate_host'] += 20
        action_scores['disable_user'] += 15
        action_scores['quarantine_file'] += 10
    elif priority == 'High':
        action_scores['block_ip'] += 15
        action_scores['isolate_host'] += 10
        action_scores['disable_user'] += 10
        action_scores['scan_host'] += 5
    elif priority == 'Medium':
        action_scores['disable_user'] += 5
        action_scores['scan_host'] += 10
        action_scores['reset_password'] += 5
    
    # Score based on Threat Intelligence (example: risk_score)
    threat_intel = enriched_alert_data.get('threat_intelligence', {})
    for ioc, intel in threat_intel.items():
        risk = intel.get('risk_score', 0)
        if risk >= 75: # High risk IoC
            action_scores['block_ip'] += 30
            action_scores['isolate_host'] += 25
            action_scores['quarantine_file'] += 15
            action_scores['disable_user'] += 10
        elif risk >= 50: # Medium risk IoC
            action_scores['block_ip'] += 15
            action_scores['isolate_host'] += 10
            action_scores['scan_host'] += 10
            action_scores['disable_user'] += 5
        
        # Check for specific malicious indicators
        if 'malicious_votes' in intel and intel['malicious_votes'] > 10:
            action_scores['block_ip'] += 10
            action_scores['isolate_host'] += 10

    # Score based on asset criticality (example: if available in enriched_alert_data)
    asset_criticality = enriched_alert_data.get('asset_criticality', 'Low')
    if asset_criticality == 'Critical':
        action_scores['isolate_host'] += 20
        action_scores['block_ip'] += 10
    elif asset_criticality == 'High':
        action_scores['isolate_host'] += 10
        action_scores['disable_user'] += 5

    # Score based on user context (example: user role)
    user_role = enriched_alert_data.get('user_role', 'Standard')
    if user_role == 'Administrator' or user_role == 'Finance':
        action_scores['disable_user'] += 15
        action_scores['reset_password'] += 10

    # --- Select Top Actions ---
    # Define thresholds or select top N actions
    # This logic can be complex, involving confidence levels and dependencies.
    
    # Example: Select actions with score > 20 and prioritize high impact ones
    selected_actions = []
    sorted_actions = sorted(action_scores.items(), key=lambda item: item[1], reverse=True)

    print(f"Action Scores: {action_scores}")

    # Simple selection logic: pick top 2-3 actions if score is significant
    for action_name, score in sorted_actions:
        if score > 20 and len(selected_actions) < 3:
            selected_actions.append(action_name)
        elif score > 10 and len(selected_actions) < 4: # Allow more if scores are lower
             selected_actions.append(action_name)

    # Ensure essential actions are considered if scores are low but context demands it
    if priority == 'Critical' and 'disable_user' not in selected_actions:
        selected_actions.append('disable_user')
    if 'block_ip' in threat_intel and threat_intel['block_ip'].get('risk_score', 0) >= 75 and 'block_ip' not in selected_actions:
         selected_actions.append('block_ip')

    # Remove duplicates and limit
    selected_actions = list(dict.fromkeys(selected_actions))[:3]

    # Map selected action names to full details and check for approval needs
    for action_name in selected_actions:
        action_details = RESPONSE_ACTIONS.get(action_name)
        if action_details:
            recommendations.append({
                'action': action_name,
                'description': action_details['name'],
                'impact': action_details['impact'],
                'requires_approval': action_details['requires_approval']
            })
            action_recommendations.append(action_details['name'])

    # Add a default ticket creation if no other actions are recommended
    if not recommendations:
        recommendations.append({
            'action': 'create_ticket',
            'description': RESPONSE_ACTIONS['create_ticket']['name'],
            'impact': RESPONSE_ACTIONS['create_ticket']['impact'],
            'requires_approval': RESPONSE_ACTIONS['create_ticket']['requires_approval']
        })
        action_recommendations.append(RESPONSE_ACTIONS['create_ticket']['name'])

    print(f"AI Recommended Actions: {action_recommendations}")
    print("--- Decision Engine Complete ---")

    return {
        'recommended_actions': recommendations,
        'final_priority': priority # Can be updated based on decision logic
    }

# --- SOAR Playbook Integration Example ---
def execute_response(enriched_alert_data):
    decision_output = decide_response_actions(enriched_alert_data)
    
    print("\n--- Executing Response Actions ---")
    print(f"Final Priority: {decision_output['final_priority']}")
    
    actions_to_execute = []
    actions_requiring_approval = []

    for action in decision_output['recommended_actions']:
        if action['requires_approval']:
            actions_requiring_approval.append(action)
        else:
            actions_to_execute.append(action['action'])
            print(f"Executing automated action: {action['description']}...")
            # In a real SOAR, this would call the appropriate API/script
            # e.g., block_ip_at_firewall(enriched_alert_data.get('source_ip'))

    if actions_to_execute:
        print(f"Successfully executed automated actions: {', '.join([a['description'] for a in actions_to_execute if a['action'] in actions_to_execute])}.")
    else:
        print("No automated actions to execute.")

    if actions_requiring_approval:
        print("\n--- Actions Requiring Human Approval ---")
        for action in actions_requiring_approval:
            print(f"- {action['description']} (Impact: {action['impact']})")
        # In a real SOAR, this would trigger a notification or task for an analyst.
        print("Please review and approve/reject these actions.")
    
    print("--- Response Execution Complete ---")
    return decision_output # Return decisions for logging/feedback

# --- Main Execution Logic ---
if __name__ == "__main__":
    # Simulate enriched alert data (output from previous steps)
    sample_enriched_alert = {
        'alert_id': 'ALERT-12345',
        'source_ip': '203.0.113.10', 
        'destination_ip': '192.168.1.50', 
        'priority': 'High',
        'asset_criticality': 'Critical', # Example context
        'user_role': 'Administrator', # Example context
        'threat_intelligence': {
            '203.0.113.10': {
                'ip_address': '203.0.113.10',
                'malicious_votes': 15,
                'suspicious_votes': 5,
                'harmless_votes': 1,
                'undetected_votes': 2,
                'country': 'Unknown',
                'as_owner': 'Example ISP',
                'tags': ['malware', 'c2'],
                'risk_score': 85
            }
        }
    }

    print(f"Enriched Alert Data: {json.dumps(sample_enriched_alert, indent=2)}")
    
    # Run the decision engine and execute response actions
    response_decisions = execute_response(sample_enriched_alert)

    print("\n--- Final Response Decisions ---")
    print(json.dumps(response_decisions, indent=2))

    print("\nIf you see this message → your lab is working perfectly!")